In [ ]:
import numpy as np
from datasets import load_dataset, get_dataset_config_names
import random
import matplotlib.pyplot as plt
from PIL import Image
import io
import sys

# ===============================================================================
# 🌟 EuroSAT RGB (유로샛 RGB) 실습 튜토리얼
# 💡 데이터셋 개요: 이 데이터셋은 지표면을 촬영한 인공위성 이미지(Sentinel-2 기반)로,
#          지형을 10가지 유형(산림, 농지, 주거지 등)으로 분류하는 작업을 위한
#          이미지 분류(Image Classification) 데이터셋입니다.
# 🚀 학습 목표: Hugging Face datasets 라이브러리를 이용해 대용량 이미지 데이터셋을
#              효율적으로 로드하고, 샘플 데이터를 분석하며, 데이터 특성에 맞는
#              '창의적인 데이터 탐색' 실습을 경험합니다.
# ===============================================================================

# --- 설정 변수 ---
DATASET_ID = "blanchon/EuroSAT_RGB"
SPLIT_NAME = "validation"  # 테스트할 분할 (Validation)
SAMPLE_COUNT = 10         # 탐색할 샘플 개수
# -----------------

print("=" * 80)
print("✨ 환영합니다! 지리정보(Geospatial) AI의 세계로 떠날 준비가 되셨나요?")
print("✨ 오늘의 미션: 위성 이미지 분류 데이터셋 탐험하기!")
print("=" * 80)

# 1. 사용 가능한 Config 확인 및 로딩 준비
try:
    configs = get_dataset_config_names(DATASET_ID)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    selected_config = configs[0]
except Exception as e:
    print(f"⚠️ Config 확인 중 오류 발생: {e}")
    selected_config = None

# 2. 데이터셋 로드 시도 (스트리밍 모드 우선!)
dataset = None
sample_data_list = []

print(f"\n--- 🚀 1단계: 데이터 로딩 (최적화된 스트리밍 방식 시도) ---")

try:
    # 1.1. 스트리밍 모드로 시도 (가장 빠르고 메모리 효율적!)
    print(f"✅ [시도 1] 스트리밍 모드 ({SPLIT_NAME})로 데이터 로드를 시도합니다...")
    dataset = load_dataset(DATASET_ID, split=SPLIT_NAME, streaming=True)
    print("✨ 성공! 스트리밍 모드에 성공적으로 접속했습니다. (대용량 데이터 처리에 최적!)")

except Exception as e:
    # 1.2. 스트리밍 실패 시 폴백 (Fallback) 처리
    print(f"\n🚨 스트리밍 모드 로드 중 오류 발생 또는 환경 제한 감지: {e.__class__.__name__}")
    print("🔄 [폴백] 스트리밍 대신, 소량의 데이터만 다운로드하여 진행하겠습니다.")
    try:
        # 스트리밍이 안될 경우, 작은 규모의 테스트 셋만 다운로드하여 진행
        dataset = load_dataset(DATASET_ID, name=DATASET_ID, split='test', streaming=False)
        print("✨ 폴백 성공! 테스트 셋을 다운로드하여 데이터를 로드했습니다.")
    except Exception as e_fallback:
        print(f"💀 치명적인 오류: 데이터 로드에 실패했습니다. {e_fallback}")
        sys.exit(1)


# 3. 데이터 샘플 추출 및 반복 처리 전략 적용
print("\n--- 🔎 2단계: 데이터 샘플 추출 및 분석 (데이터셋 전체를 탐험하기 어렵다면, 일부만 봅니다!) ---")

# 스트리밍 패턴 처리 로직 (필수 규칙 준수)
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print(f"✅ 스트리밍 데이터셋 감지! 상위 {SAMPLE_COUNT}개만 샘플링하여 로드합니다.")
    # list()로 전체 샘플을 메모리에 미리 로드합니다.
    sample_data_list = list(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)
    print(f"✅ 일반 데이터셋 감지! 상위 {SAMPLE_COUNT}개만 샘플링하여 로드합니다.")
    # list()로 전체 샘플을 메모리에 미리 로드합니다.
    sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))


# 4. 🎨 창의적인 실습 예시: 클래스 분포 분석 및 시각화
print(f"\n--- ✨ 3단계: 탐험 및 시각화 (총 {len(sample_data_list)}개의 샘플 분석) ---")

# 4.1. 클래스 분포 분석 (통계적 분석)
print("\n[📊 1. 클래스 분포 분석 (Label Frequency)]")
class_counts = {}
label_names = [
    "Annual Crop", "Forest", "Herbaceous Vegetation", "Highway", 
    "Industrial Buildings", "Pasture", "Permanent Crop", "Residential Buildings", 
    "River", "SeaLake"
]

for sample in sample_data_list:
    # 'label' 필드에서 클래스 이름을 가져와서 카운트합니다.
    label_name = sample['label']
    if label_name in class_counts:
        class_counts[label_name] += 1
    else:
        class_counts[label_name] = 1

# 분포 결과를 보기 좋게 출력합니다.
sorted_counts = dict(sorted(class_counts.items(), key=lambda item: item[1], reverse=True))
for name, count in sorted_counts.items():
    print(f"    -> {name:<25}: {count} 회")
print("\n💡 튜터 Tip: 이 카운트는 랜덤하게 뽑힌 소수의 샘플을 기반한 근사치입니다. 전체 데이터셋의 균형을 예측하는 재미있는 과정이죠!")


# 4.2. 이미지 시각화 (AI가 실제로 보는 방식 체험)
print("\n[🖼️ 2. 이미지 샘플 시각화 (Visualization)]")

if sample_data_list:
    # 시각화할 이미지와 레이블을 분리합니다.
    images = []
    labels_for_plot = []

    for sample in sample_data_list:
        # 이미지 데이터를 PIL Image 객체로 변환합니다.
        image = sample['image']
        images.append(image)
        # 레이블 이름은 문자열로 사용합니다.
        labels_for_plot.append(sample['label'])

    # Matplotlib 설정 및 그리기
    fig, axes = plt.subplots(2, 5, figsize=(15, 6)) # 2행 5열 배치 (샘플 개수만큼 유동적)
    axes = axes.flatten() # 2차원 배열을 1차원으로 펼쳐서 반복 처리 쉽게 만듦

    for i in range(min(len(images), 10)): # 최대 10개까지만 표시
        ax = axes[i]
        img = images[i]
        label = labels_for_plot[i]

        # 이미지 표시 (주의: 이미지 크기가 클 수 있으므로, Matplotlib가 적절히 스케일링합니다.)
        ax.imshow(img)
        ax.set_title(f"Class: {label}", fontsize=10)
        ax.axis('off') # 축 숨기기

    # 남는 축은 숨겨서 깔끔하게 만듭니다.
    for i in range(min(len(images), 10), len(axes)):
        fig.delaxes(axes[i])

    plt.suptitle(f"EuroSAT RGB Sample Exploration (Total Samples: {len(sample_data_list)})", y=1.02)
    plt.show()
    print("\n🎉 멋진 시각화가 완료되었습니다! 이제 데이터의 패턴이 보이시나요?")
else:
    print("❌ 샘플 데이터를 가져오는 데 실패하여 시각화 작업을 건너뜁니다.")

print("\n" + "=" * 80)
print("👏 축하합니다! 당신은 Hugging Face의 대용량 AI 데이터셋을 성공적으로 탐험했습니다.")
print("✨ 이 과정을 통해 대용량 데이터 처리(Streaming), 데이터 구조 이해, 그리고 기초 EDA 능력을 키웠습니다.")
print("💡 다음 단계는 이 구조를 활용하여 CNN 모델을 훈련하는 것입니다!")
print("=" * 80)